In [1]:
# Experiment
# - Info-Gain SoRL
# - 2 Phase switch, Phase I: baseline language modeling, Phase II: SoRL with info-gain target & reward scaling.

import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

# 1. load a small subset of the dataset (checked)
# 2. tokenize & build data loader (checked)
# 3. train with SoRL. 
# 4. visualize abstraciton dynamics (similar to copy-n-paste, but more generally on per-n-gram statistics)

In [2]:
from data.tinystory_local import TinyStoriesDataLoader, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 32
K = max_len // 2

loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device=model.device)

batch_size = 8
memory_span = 2 * max_len + 2
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

Loading 100 stories from TinyStories train...
Loaded 100 stories, 3200 tokens total, 0.01 MB
Collected 199 unique 16-chunks


In [3]:
# Idea #1. 
# -> Coarse-to-fine abstraction 
# Q1. Is 'coarse' abstraction transferable to 'fine' abstraction? 

In [ ]:
# train SoRL with interleaved phase switch (phase I: baseline lm | phase II: SoRL with info-gain target & reward scaling)
from sorl.neo_utils import sorl_evaluate_v2, sorl_search_v8
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss_v8 

# --- orthogonal initialization on abs param --- 
# orthogonalize_abs_param(model, do_wte=True, do_head=True)

loss_fn = SoRLLoss_v8(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 200
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    # Issue -> this guy padds lot of BOS_TOKEN_ID on each data point, I don't want padding
    tokens, doc_ids = loader.get_batch(batch_size)

    with torch.no_grad(): 
        best_data, best_ppt, search_adv, utility_reward = sorl_search_v8(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss --- 
    base_traj_loss = model.forward(tokens, memory_span, attn_blocksize)[0].mean()
    info_loss, abs_loss, zipf_bigram_loss = loss_fn(best_data, model, base_traj_loss.detach(), utility_reward, memory_span, attn_blocksize)    
    loss = base_traj_loss + alpha_info_gain * info_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    # ---- search info gain ---
    rel_info_gain = ((-info_loss) / base_traj_loss).detach()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss, abs_logits, abs_tokens, avg_logit_sim = sorl_evaluate_v2(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)

            _, _, zipf_bigram_loss = loss_fn(val_tokens, model, base_traj_loss, utility_reward, memory_span, attn_blocksize)
            abs_stats.update(abs_logits, traj_loss, rel_info_gain, abs_tokens, doc_ids)
            record['vocab_util'].append(abs_stats.vocab_util * 100)
            record['greedy_adv'].append(val_adv.item() * 100)
            record['abs_loss'].append(abs_loss.mean().item())
            record['traj_loss'].append(traj_loss.mean().item())
            record['base_traj_loss'].append(base_traj_loss.item())
            record['bigram_rep_rate'].append(abs_stats.bigram_rep_rate)
            record['kl_soft_zipf'].append(zipf_bigram_loss.item())
            record['rel_search_info_gain'].append(rel_info_gain)

        print(f"\nstep {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f}")
        
        img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        img_frames.append(img)
        # break


step 0 | base traj loss: 10.19 | cond traj loss: 10.74 | rel search info gain: -6.25% | greedy adv: 0.02% | vocab util: 43.75%  | avg logit sim: 0.75 |  bigram-zipf kl: 62.90 | bigram rep rate: nan | rel info gain (search): -0.06

step 2 | base traj loss: 10.08 | cond traj loss: 10.58 | rel search info gain: -6.24% | greedy adv: 0.02% | vocab util: 56.25%  | avg logit sim: 0.90 |  bigram-zipf kl: 32.86 | bigram rep rate: nan | rel info gain (search): -0.06

step 4 | base traj loss: 9.85 | cond traj loss: 10.27 | rel search info gain: -6.27% | greedy adv: 0.01% | vocab util: 56.25%  | avg logit sim: 0.99 |  bigram-zipf kl: 18.10 | bigram rep rate: nan | rel info gain (search): -0.06

step 6 | base traj loss: 9.54 | cond traj loss: 9.92 | rel search info gain: -6.26% | greedy adv: 0.00% | vocab util: 62.50%  | avg logit sim: 1.00 |  bigram-zipf kl: 10.66 | bigram rep rate: nan | rel info gain (search): -0.06

step 8 | base traj loss: 9.21 | cond traj loss: 9.58 | rel search info gain: -

In [12]:
best_data

tensor([[50256,  7454,   612,   547,   734,  2460,  3706,  5811,   290,  3409,
            13,  1119,   547,  2712,  1978,   287,   262, 50260, 11376,   530,
          1110,    13,  3409,  6497,   257, 15061,   290,   788,  6235,   284,
           257,  2042,  6512, 50256,  3198,  1110,    11,   257,  3049,  4639,
          3706,  5045,  1816,   329,   257,  6594,   287,   465,  7812,  1097,
         50272,    13,   679,  6151,   284,  2866,   866,   262,  4675,   290,
          1254,   262,  2344,   287,   465,  4190, 50256,  7454,   612,   373,
           257,  2933,  3706,  4186,    13,  4186,  2227,   284,   766,  1223,
          2041,    11,   523, 50268,   339,  1816,   284,   257,  1029,  8598,
            13,   632,   373, 27737,   290,  5814,  2354,   523,   262, 50256,
          7454,   612,   547,   734,  2460,  3706,  5811,   290,  3409,    13,
          1119,   547,  2712,  1978,   287,   262, 50266, 11376,   530,  1110,
            13,  3409,  6497,   257, 15061,   290,  

In [ ]:
# exp 1. I use one forward propagation to compute base traj ppt & base traj loss
# -> but this produces a differetn training mechanism, where base traj loss lags behind
# --- base traj loss: 3.20 | cond traj loss: 1.43 | rel search info gain: >50%

# exp 2. doc-level info gain (w=1.0) | 400 steps
# --- base traj loss: 1.11 | cond traj loss: 1.19 | rel search info gain: -1.8%

# exp 4. doc-level info gain (w=1.0) | 1200 steps 
# ---- base traj loss: 0.18 | cond traj loss: 0.18 | greedy adv: 18% | vocab util: 81%
#      n=10, search info gain: 6.3%
# (obs). mid training has collapse of rel info gain & greedy adv. 
#        instability of training signal might be the culprit here

# exp 5. doc-level info gain (w=10.0) | 400 steps
# (obs). vocab collapse first, then gradually increase, but no 'crash' occurs (stability high)
# --- base traj loss: 1.29 | cond traj loss: 0.92 | rel search info gain: 25% | vocab util: 25% 
#     n=10, search info gain: 26.6% | n=5, rel search info gain: 26.26% 

# exp 6. doc-level info gain (w=10.0) | 1200 steps
# --- base traj loss: 0.18 | cond traj loss: 0.15 | rel search info gain: 17.94% | vocab util: 56.25%

# exp 7. doc-level info gain (w=5.0) | 1200 steps
# --- base traj loss: 0.166 | cond traj loss: 0.148 | rel search info gain: 13.0% | vocab util: 43.75% 

# exp 8. batch-level info gain loss (w=10.0) | 400 steps
# ---- base traj loss: 0.93 | cond traj loss: 0.68 | rel search info gain: 30% | vocab util: 50% 

# exp 9. batch-level info gain loss (w=10.0) & utility reward scaling | 400 steps
# ---- base traj loss: 1.28 | cond traj loss: 0.75 | rel search info gain: 36% | vocab util: 56.25% 
# (trait). cond traj loss beats base traj loss from the start

# Full tinystories run gives negative rel search info gain, but base traj loss matches with baseline runs. 

In [14]:
from sorl.forget import compute_abs_stats

n = 5
temperature = torch.tensor([0.0] + [5.0] * (n - 1))

# ---- statistics on info gain reward ---
i = 0
batch_size = 16
ig = []
while i < len(loader.stories): 
    batch_indices = torch.arange(i, min(i + batch_size, len(loader.stories)))
    loader.get_specific(batch_indices) 
    i += batch_size
    tokens, doc_ids = loader.get_specific(batch_indices) 
    traj_loss, abs_logits, abs_tokens, doc_rel_info_gain, rel_info_gain, _, _ = compute_abs_stats(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature,
                                                                                        truncate_seq_len=False)
    # abs_stats.update(abs_logits, traj_loss, doc_rel_info_gain, abs_tokens, doc_ids)
    ig.append(rel_info_gain)
    # break

 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 16 | doc count: 16
 ---- base doc count: 4 | doc count: 4


In [16]:
torch.tensor(ig).mean()

/var/folders/vx/6bqm9pg11yl1j_t5rgg9zczm0000gn/T/ipykernel_7884/734922377.py:1: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  torch.tensor(ig).mean()


tensor(0.3054)

In [8]:
# n=1, rel info gain: 6.3%
# n=2, rel info gain: 6.3% 
# n=10, rel info gain: 6.6%
# with 1200 steps (no reset on abs param) policy lag is much less of a problem
# and we get a stable 6% improvement in perplexity
# -> if we further allow dynamic 'inclusion' of abstract tokens, we get even better result but anyway ...
torch.tensor(ig).mean()

/var/folders/vx/6bqm9pg11yl1j_t5rgg9zczm0000gn/T/ipykernel_21500/3672364059.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  torch.tensor(ig).mean()


tensor(0.1301)

In [12]:
# rel_info_gain
torch.tensor(ig).mean()

# (Scaling test-time compute in abstract space --> "thinking is acting in an imaginary space")
# n=1, t=[0] -> relative info gain: 0.59% 
# n=2, t=[0, 5] -> relative info gain: 3.6% 
# n=5, t=[0, 5, ..., 5] -> relative info gain: 6.08%
# n=8, t=[0, 5, ..., 5] -> relative info gain: 4.8%
# n=10, t=[0, 5, ..., 5] -> relative info gain: 7.26%
# n=20, t=[0, 5, ..., 5] -> relative info gain: 6.7% 
# n=30, t=[0, 5, ..., 5] -> relative info gain: 7.55%



/var/folders/vx/6bqm9pg11yl1j_t5rgg9zczm0000gn/T/ipykernel_14596/1434340856.py:2: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  torch.tensor(ig).mean()


tensor(0.0632)

In [ ]:
# Hypothesis #1. 
# - if SoRL does have a 'schema effect', then the more organized it gets (later ckpt) will have 
#   (1). better learning efficiency
#                            (for this, we might need to use some 'new' stories data point)
#   (2). lesser interference (% of closeness->interference correlation should be lesser) 
#                            (% of closeness->alignment correlation should be higher)

# Question #1. 
# Does SoRL has advantage over baseline in info gain and continual learning? 

# Question #2. 
# Policy lag leads to "search adv" & "greedy adv" gap. resolving this and we'll get a better policy model
# together with a better "greedy adv", the ability to manipulate the inner-monologue can be improved. 

# Question #3. 
# Can we replicate the improvement on big-scale dataset?

# Obs #1. 



In [ ]:


# exp 7. base traj loss included SoRL (with online info gain loss & reward scaling)
# (1). base traj loss and cond traj loss improve simultaneously
# ---- base traj loss: 1.01 | cond traj loss: 1.08 | greedy adv: 18.9% | vocab util: 81.25%
# (2). something about the ability to push both p(s) and p(s | a) up is attractive to me
# Obs #1. During training, the searched abstraction has information gain p(s | a) > p(s)
#         but such advantage doesn't exist in greedy policy rollouts p(s | a*) < p(s)
# Hyp #1. If we train till saturation, we should observe p(s | a*) > p(s)?


# exp 8. base traj loss included SoRL (with online info gain loss & reward scaling), 3x compute
# ---- base traj loss: 0.18 | cond traj loss: 0.19 | greedy adv: 18% | vocab util: 81%
# ---- rel info gain (search): 15.4% (t1=0, t2=5.0)



# Obs #1. We should however note that exp 3 & exp 4 has matching performance. 
#         One prev issue is that SoRL doesn't match baseline in p(s) in TinyStories dataset. 

# Issue #1. How do we avoid the degradation on greedy adv? 

# Idea #1. 
# -> instead of interleaving, how about we combine loss function? 


# base_traj_loss = model.forward(tokens, memory_span, attn_blocksize)[0].mean()
# traj_loss, abs_loss, zipf_bigram_loss, topo_loss = loss_fn(best_data, model, base_traj_ppt, memory_span, attn_blocksize, info_gain_reward)
# some sort of weighted sum of 2 terms ... (we want to encourage info gain & better base traj)



In [44]:
from sorl.forget import collect_forget_data, plot_normalized_correlation_lines, train_forget_vec
from tqdm import tqdm as tqdm

# ---- Memory Interference Experiment ---

# --- Load model, prepare abs stats record ---
model.load_state_dict(torch.load('sorl_tinystories.pt'))

abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

abs_stats_post = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# # ---- Compute 'Forget Matrix' --- 
# forget_mat = torch.zeros(num_stories, num_stories, device=model.device)
# num_steps = 40
# for train_idx in tqdm(range(num_stories)): 
#     forget_vec = train_fogret_vec(train_idx, loader, model, abs_stats, abs_stats_post, optimizer, num_steps,
#                                 max_iterations, memory_span, attn_blocksize, temperature, K, r_min, reward_mode, loss_fn, alpha_abs, alpha_soft_zipf, alpha_topo,
#                                 ckpt_path="sorl_tinystories.pt")
#     forget_mat[train_idx] = forget_vec 

# torch.save(forget_mat, "forget_mat.pt") # save it just in case

# # ---- Visualize 'Forget Matrix' --- 
# correlation_data, ham_corrs = collect_forget_data(forget_mat, abs_stats)

# plot_normalized_correlation_lines(
#     correlation_data, 
#     ham_corrs,
#     xlabel="Abstraction Edit Distance", 
#     ylabel="Forgetting (Δ Perplexity)",
#     title="Distant Abstractions → Less Forgetting (TinyStories)"
# )

In [ ]:
# Exp #1. 
# 'self-organization' of abstraction representation --> will SoRL tries to pull mixing memory apart? 
# Exp #2. 
# setting up tiny memory dataset training pipeline 
# Question #1. 
# What if we use all-rollout SoRL with utility advantage + KL reg terms? 


In [ ]:
from data.tinystory_local import *
from sklearn.decomposition import PCA

cs_sim = abs_stats.compute_cross_doc_logit_sim()
# cs_sim = abs_stats.compute_cross_doc_hamming()
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(cs_sim.cpu().numpy())

# Now use in 3D visualization
train_idx = 0
perplexity = forget_mat[train_idx]
img = visualize_forget_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)
# img = visualize_perplexity_terrain(coords_2d, perplexity, trained_idx=train_idx, step=step)

In [5]:
if len(img_frames) > 0:
    img_frames[0].save(
        'tinystories_dynamics (SoRL with 10.0 info gain loss + utility reward scaling).gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=200,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

Saved GIF with 600 frames


In [ ]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

idx = tokens[:, :15].clone()

img_frames = []
for i in range(30): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds per frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")

In [ ]:
# Request #1. 
# -> Full scale experiment on TinyStories & FineWeb

In [ ]:
# marg cond w = 1.0 
# traj_loss: 0.85 | abs_loss: 0.04 | search adv: 21.01% | vocab util: 93.75%  | marg_ent: 2.67 | cond_ent: 0.05 | avg cos sim: 0.44

# soft bigram zipf kl w = 1.0 
# traj_loss: 0.89 | abs_loss: 0.53 | search adv: 25.84% | vocab util: 68.75%  | kl_soft_zipf: 0.24 | avg cos sim: 0.47
# -> visually I observe much less 'repetitions'

# soft bigram zipf kl w = 1.0 & utility reward scaling r = max(p(s|a)/p(s), 1.0)
# traj_loss: 0.79 | abs_loss: 0.40 | search adv: 30.13% | vocab util: 62.50%  | kl_soft_zipf: 0.17 | avg cos sim: 0.44 | topo sim: 0.77 

# soft bigram zip w=1.0 & utility reward scaling & topo reg w=1.0
# traj loss: 1.50 | search adv: 20% | avg cos sim: 0.24 | topo sim: 0.53
# => degrades utility, no go

